# MLB Subsequent-Season bWAR Forecasting — End-to-End Pipeline

This notebook contains the **final dissertation implementation only**. Earlier experimental cells, duplicate TFT conversion code, diagnostic-only cells, and the incorrect batch-averaged TFT RMSE comparison have been removed.

## Data note
The dissertation used **locally stored annual Statcast CSV files for 2015–2023** and local Baseball-Reference WAR files. The raw data are intentionally not included in the repository because of their size. This notebook begins from those local raw files; it does **not** download Statcast data.

## Final modelling design
- Predictors: Statcast-derived and traditional performance measures
- Target: subsequent-season Baseball-Reference bWAR
- Monthly sequence: April–September (6 time steps)
- Training-only feature selection: 26 → 20 predictors
- Chronological split: 2015–2020 train, 2021–2022 validation, 2023 test
- Models: Ridge Regression, Random Forest, LSTM, Temporal Fusion Transformer
- Interpretability: LSTM SHAP and TFT native variable-selection / attention outputs

Run this notebook from the `notebooks/` directory so paths such as `../data/raw/` resolve correctly.


## 1. Data preparation and cross-database alignment


In [ ]:
# ============================================================
# FINAL DATA PREPARATION
#
# IMPORTANT:
# The dissertation analysis uses locally stored annual Statcast CSV files
# (2015–2023) plus locally stored Baseball-Reference WAR files.
# Raw data are NOT downloaded inside this notebook.
#
# Expected files:
# ../data/raw/statcast_2015_full.csv ... statcast_2023_full.csv
# ../data/raw/war_daily_bat.txt
# ../data/raw/war_daily_pitch.txt
# ============================================================

import pandas as pd
import numpy as np
import pybaseball as pyb
import os
import json

# Disable caching and warnings
pyb.cache.config.enabled = False
pd.options.mode.chained_assignment = None

def get_bwar_and_mapping_local(target_year, bat_file_path, pitch_file_path):
    print(f"🔄 Loading batter and pitcher bWAR data for {target_year}...")
    try:
        df_bat = pd.read_csv(bat_file_path)
        df_pitch = pd.read_csv(pitch_file_path)
    except Exception as e:
        print(f"❌ Failed to read local files: {e}")
        return None
        
    df_bat_year = df_bat[df_bat['year_ID'] == target_year][['player_ID', 'name_common', 'WAR']]
    df_pitch_year = df_pitch[df_pitch['year_ID'] == target_year][['player_ID', 'name_common', 'WAR']]
    df_war_year = pd.concat([df_bat_year, df_pitch_year], ignore_index=True)
    
    df_chadwick = pyb.chadwick_register()
    df_map = df_chadwick[['key_mlbam', 'key_bbref']].dropna()
    df_map['key_mlbam'] = df_map['key_mlbam'].astype(int)
    
    df_war_mapped = pd.merge(df_war_year, df_map, left_on='player_ID', right_on='key_bbref', how='inner')
    df_war_mapped = df_war_mapped[['key_mlbam', 'name_common', 'WAR']]
    df_war_mapped.rename(columns={'key_mlbam': 'player_id', 'name_common': 'Name'}, inplace=True)
    df_war_mapped = df_war_mapped.dropna(subset=['WAR'])
    df_war_mapped = df_war_mapped.groupby(['player_id', 'Name'], as_index=False)['WAR'].sum()
    
    return df_war_mapped

def prepare_sequence_data_combined(df_statcast, df_war, valid_months=[4, 5, 6, 7, 8, 9]):
    print("⏳ Performing time-binning to extract advanced physical and traditional features...")
    
    df_statcast['game_date'] = pd.to_datetime(df_statcast['game_date'])
    df_statcast['month'] = df_statcast['game_date'].dt.month
    df_sc_filtered = df_statcast[df_statcast['month'].isin(valid_months)].copy()
    
    # ==========================================
    # Extract traditional statistics from Statcast events
    # ==========================================
    # Normalize events to lowercase strings for consistent processing
    df_sc_filtered['events'] = df_sc_filtered['events'].astype(str).str.lower()
    
    # Traditional batter statistics (binary indicators; sum gives event counts)
    df_sc_filtered['trad_HR'] = (df_sc_filtered['events'] == 'home_run').astype(int)
    df_sc_filtered['trad_BB'] = (df_sc_filtered['events'] == 'walk').astype(int)
    df_sc_filtered['trad_Hit'] = df_sc_filtered['events'].isin(['single', 'double', 'triple', 'home_run']).astype(int)
    
    # Traditional pitcher statistics
    df_sc_filtered['trad_SO'] = (df_sc_filtered['events'] == 'strikeout').astype(int)
    df_sc_filtered['trad_HR_allowed'] = (df_sc_filtered['events'] == 'home_run').astype(int)
    df_sc_filtered['trad_BB_allowed'] = (df_sc_filtered['events'] == 'walk').astype(int)

    # --- 1. Batter advanced + traditional feature aggregation ---
    bat_agg_dict = {
        'launch_speed': ['mean', 'max'],
        'launch_angle': ['mean'],
        'hit_distance_sc': ['mean', 'max'],
        'estimated_woba_using_speedangle': ['mean'],
        'delta_run_exp': ['sum', 'mean'],
        'release_speed': ['mean'],
        'pfx_x': ['mean'],
        'pfx_z': ['mean'],
        # Aggregate traditional statistics by sum
        'trad_HR': ['sum'],
        'trad_BB': ['sum'],
        'trad_Hit': ['sum']
    }
    df_bat = df_sc_filtered.groupby(['batter', 'month']).agg(bat_agg_dict)
    df_bat.columns = [f'bat_{col[0]}_{col[1]}' for col in df_bat.columns]
    df_bat = df_bat.reset_index().rename(columns={'batter': 'player_id'})
    
    # --- 2. Pitcher advanced + traditional feature aggregation ---
    pit_agg_dict = {
        'release_speed': ['mean', 'max'],
        'release_spin_rate': ['mean'],
        'release_extension': ['mean'],
        'pfx_x': ['mean'],
        'pfx_z': ['mean'],
        'estimated_woba_using_speedangle': ['mean'],
        'delta_run_exp': ['sum', 'mean'],
        # Aggregate traditional statistics by sum
        'trad_SO': ['sum'],
        'trad_HR_allowed': ['sum'],
        'trad_BB_allowed': ['sum']
    }
    df_pit = df_sc_filtered.groupby(['pitcher', 'month']).agg(pit_agg_dict)
    df_pit.columns = [f'pit_{col[0]}_{col[1]}' for col in df_pit.columns]
    df_pit = df_pit.reset_index().rename(columns={'pitcher': 'player_id'})
    
    # --- 3. Merge and align ---
    df_agg = pd.merge(df_bat, df_pit, on=['player_id', 'month'], how='outer')
    feature_cols = [col for col in df_agg.columns if col not in ['player_id', 'month']]
    
    sc_players = df_agg['player_id'].unique()
    war_players = df_war['player_id'].unique()
    valid_players = np.sort(np.intersect1d(sc_players, war_players))
    
    idx = pd.MultiIndex.from_product([valid_players, valid_months], names=['player_id', 'month'])
    df_grid = pd.DataFrame(index=idx).reset_index()
    
    df_merged = pd.merge(df_grid, df_agg, on=['player_id', 'month'], how='left')
    df_merged[feature_cols] = df_merged[feature_cols].fillna(0)
    df_merged = df_merged.sort_values(by=['player_id', 'month'])
    
    X = df_merged[feature_cols].values.reshape(len(valid_players), len(valid_months), len(feature_cols))
    
    df_war_sorted = df_war.set_index('player_id').reindex(valid_players).reset_index()
    y = df_war_sorted['WAR'].values
    names = df_war_sorted['Name'].values
    
    return X, y, names, feature_cols

if __name__ == "__main__":
    FEATURE_YEARS = list(range(2015, 2024)) 
    DATA_DIR = "../data/raw"
    OUTPUT_DIR = "../data/processed"
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    BAT_FILE_PATH = os.path.join(DATA_DIR, "war_daily_bat.txt")
    PITCH_FILE_PATH = os.path.join(DATA_DIR, "war_daily_pitch.txt")
    
    all_X, all_y, all_names, all_years = [], [], [], []
    
    for feature_year in FEATURE_YEARS:
        target_year = feature_year + 1
        raw_data_path = os.path.join(DATA_DIR, f"statcast_{feature_year}_full.csv")
        
        if os.path.exists(raw_data_path) and os.path.exists(BAT_FILE_PATH) and os.path.exists(PITCH_FILE_PATH):
            print(f"\n📊 Processing: {feature_year} features -> {target_year} bWAR")
            df_raw = pd.read_csv(raw_data_path, low_memory=False)
            df_war = get_bwar_and_mapping_local(target_year, BAT_FILE_PATH, PITCH_FILE_PATH)
            
            X_year, y_year, names_year, features = prepare_sequence_data_combined(df_raw, df_war)

            all_X.append(X_year)
            all_y.append(y_year)
            all_names.extend(names_year)

            # Store the feature year for each sample
            all_years.append(
                np.full(len(y_year), feature_year, dtype=np.int16)
            )
    
    if all_X:
        X_final = np.concatenate(all_X, axis=0)
        y_final = np.concatenate(all_y, axis=0)
        
        np.save(os.path.join(OUTPUT_DIR, 'X_final.npy'), X_final)
        np.save(os.path.join(OUTPUT_DIR, 'y_final.npy'), y_final)
        with open(os.path.join(OUTPUT_DIR, 'feature_names.json'), 'w') as f:
            json.dump(features, f)
            
        print("\n🏆 Multi-year preprocessing complete (traditional statistics integrated)!")


## 2. Build predictor-year labels


In [ ]:
# ============================================================
# SAVE SAMPLE YEARS FOR TEMPORAL VALIDATION
# ============================================================

import numpy as np
import os

OUTPUT_DIR = "../data/processed"

# Combine the year labels accumulated during preprocessing
sample_years = np.concatenate(all_years, axis=0)

# Verify exact alignment with the latest dataset
assert len(sample_years) == len(X_final) == len(y_final)

# Save
np.save(
    os.path.join(OUTPUT_DIR, "sample_years.npy"),
    sample_years
)

print("=" * 70)
print("✅ sample_years created successfully")
print(f"Total samples: {len(sample_years)}")

print("\n📅 Sample year distribution:")
unique_years, counts = np.unique(sample_years, return_counts=True)

for year, count in zip(unique_years, counts):
    print(f"{year}: {count} samples")

print("\n💾 Saved:")
print(os.path.join(OUTPUT_DIR, "sample_years.npy"))
print("=" * 70)

## 3. Training-only feature selection


In [ ]:
# ============================================================
# TRAIN-ONLY FEATURE PRUNING
# Feature selection is determined exclusively using
# the training years (2015–2020).
# Validation and test data are not involved in feature selection.
# ============================================================

import numpy as np
import pandas as pd
import json
import os

INPUT_DIR = "../data/processed"
CORR_THRESHOLD = 0.05

TRAIN_YEARS = [2015, 2016, 2017, 2018, 2019, 2020]

print("=" * 70)
print("TRAIN-ONLY Feature Pruning")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load the complete processed dataset
# ------------------------------------------------------------

X_final = np.load(
    os.path.join(INPUT_DIR, "X_final.npy")
)

y_final = np.load(
    os.path.join(INPUT_DIR, "y_final.npy")
)

sample_years = np.load(
    os.path.join(INPUT_DIR, "sample_years.npy")
)

with open(
    os.path.join(INPUT_DIR, "feature_names.json"),
    "r"
) as f:
    feature_names = json.load(f)

assert len(X_final) == len(y_final) == len(sample_years)

# ------------------------------------------------------------
# 2. Restrict feature selection to the training years
# ------------------------------------------------------------

feature_selection_mask = np.isin(
    sample_years,
    TRAIN_YEARS
)

X_feature_selection = X_final[feature_selection_mask]
y_feature_selection = y_final[feature_selection_mask]

print(
    f"Feature-selection years: "
    f"{sorted(np.unique(sample_years[feature_selection_mask]))}"
)

print(
    f"Feature-selection samples: "
    f"{len(X_feature_selection)}"
)

# ------------------------------------------------------------
# 3. Calculate feature correlations using training data only
# ------------------------------------------------------------

selected_indices = []
selected_features = []

for i, feat_name in enumerate(feature_names):

    # Treat zero-padded observations as missing values
    masked_feat = np.ma.masked_equal(
        X_feature_selection[:, :, i],
        0
    )

    # Calculate the season-level mean for each player-season
    feat_season_mean = (
        masked_feat
        .mean(axis=1)
        .filled(np.nan)
    )

    # Retain observations with valid feature and target values
    valid_mask = (
        np.isfinite(feat_season_mean)
        & np.isfinite(y_feature_selection)
    )

    if (
        valid_mask.sum() > 50
        and np.std(feat_season_mean[valid_mask]) > 0
    ):
        r = np.corrcoef(
            feat_season_mean[valid_mask],
            y_feature_selection[valid_mask]
        )[0, 1]

        if not np.isfinite(r):
            r = 0.0
    else:
        r = 0.0

    if abs(r) >= CORR_THRESHOLD:

        selected_indices.append(i)
        selected_features.append(feat_name)

        print(
            f"RETAINED: {feat_name:40s} "
            f"| Train r = {r:.3f}"
        )

    else:

        print(
            f"REMOVED:  {feat_name:40s} "
            f"| Train r = {r:.3f}"
        )

# ------------------------------------------------------------
# 4. Apply the features selected from the training data
#    unchanged to the complete dataset
#
# Note:
# Validation and test observations are only subsetted here.
# They do not contribute to the feature-selection procedure.
# ------------------------------------------------------------

X_golden = X_final[:, :, selected_indices]

print("\n" + "=" * 70)

print(
    f"Training-only pruning retained "
    f"{len(selected_indices)} of {len(feature_names)} features."
)

print(
    f"Full X_golden shape: {X_golden.shape}"
)

print("\nFinal selected features:")

for i, feat in enumerate(selected_features, 1):
    print(f"{i:2d}. {feat}")

print("=" * 70)

## 4. Save the canonical modelling dataset


In [ ]:
# ============================================================
# FINAL CANONICAL DATASET
# All downstream models use this canonical dataset
# ============================================================

import os
import json
import numpy as np

FINAL_DIR = "../data/processed/final"
PROCESSED_DIR = "../data/processed"

os.makedirs(FINAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Final feature tensor + target
# ------------------------------------------------------------
X_model = X_golden.copy()
y_model = y_final.copy()
final_feature_names = selected_features.copy()

# ------------------------------------------------------------
# 2. Load the feature year associated with each sample
# ------------------------------------------------------------
model_years = np.load(
    os.path.join(PROCESSED_DIR, "sample_years.npy")
)

# ------------------------------------------------------------
# 3. Consistency checks
# ------------------------------------------------------------
assert X_model.ndim == 3
assert X_model.shape[0] == len(y_model)
assert X_model.shape[0] == len(model_years)
assert X_model.shape[2] == len(final_feature_names)

print("=" * 70)
print("✅ FINAL MODEL DATASET created")

print(f"Samples      : {X_model.shape[0]}")
print(f"Time steps   : {X_model.shape[1]}")
print(f"Features     : {X_model.shape[2]}")
print(f"Target shape : {y_model.shape}")

print("\n📅 Year distribution:")
for year in np.unique(model_years):
    print(
        f"{year}: {(model_years == year).sum()} samples"
    )

print("\nFinal features:")
for i, feat in enumerate(final_feature_names, 1):
    print(f"{i:2d}. {feat}")

# ------------------------------------------------------------
# 4. Save canonical dataset
# ------------------------------------------------------------
np.save(
    os.path.join(FINAL_DIR, "X_model.npy"),
    X_model
)

np.save(
    os.path.join(FINAL_DIR, "y_model.npy"),
    y_model
)

np.save(
    os.path.join(FINAL_DIR, "model_years.npy"),
    model_years
)

with open(
    os.path.join(FINAL_DIR, "final_feature_names.json"),
    "w"
) as f:
    json.dump(final_feature_names, f, indent=2)

print("\n💾 Final dataset saved")
print("=" * 70)

## 5. Chronological train / validation / test split


In [ ]:
# ============================================================
# UNIFIED CHRONOLOGICAL TEMPORAL SPLIT
# The same chronological split is applied to all models.
# Train: 2015–2020
# Validation: 2021–2022
# Test: 2023
# ============================================================

import numpy as np

TRAIN_YEARS = [2015, 2016, 2017, 2018, 2019, 2020]
VAL_YEARS   = [2021, 2022]
TEST_YEARS  = [2023]

print("=" * 70)
print("UNIFIED CHRONOLOGICAL TEMPORAL SPLIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create chronological masks based on predictor year
# ------------------------------------------------------------

train_mask = np.isin(sample_years, TRAIN_YEARS)
val_mask   = np.isin(sample_years, VAL_YEARS)
test_mask  = np.isin(sample_years, TEST_YEARS)

# ------------------------------------------------------------
# 2. Apply the split to the common modelling tensor and target
# ------------------------------------------------------------

X_train = X_golden[train_mask]
X_val   = X_golden[val_mask]
X_test  = X_golden[test_mask]

y_train = y_final[train_mask]
y_val   = y_final[val_mask]
y_test  = y_final[test_mask]

years_train = sample_years[train_mask]
years_val   = sample_years[val_mask]
years_test  = sample_years[test_mask]

# ------------------------------------------------------------
# 3. Basic integrity checks
# ------------------------------------------------------------

assert len(X_train) == len(y_train) == len(years_train)
assert len(X_val)   == len(y_val)   == len(years_val)
assert len(X_test)  == len(y_test)  == len(years_test)

assert (
    len(X_train) + len(X_val) + len(X_test)
    == len(X_golden)
)

# ------------------------------------------------------------
# 4. Print split summary
# ------------------------------------------------------------

print(f"Training years:   {sorted(np.unique(years_train))}")
print(f"Validation years: {sorted(np.unique(years_val))}")
print(f"Test years:       {sorted(np.unique(years_test))}")

print("\nSample counts:")
print(f"Train:      {len(X_train)}")
print(f"Validation: {len(X_val)}")
print(f"Test:       {len(X_test)}")
print(f"Total:      {len(X_golden)}")

print("\nTensor shapes:")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

print("\nTarget shapes:")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"y_test shape:  {y_test.shape}")

print("=" * 70)
print("Chronological split completed successfully.")
print("=" * 70)

## 6. Build real static covariates for TFT


In [ ]:
# ============================================================
# BUILD REAL STATIC FEATURES
# Reconstruct MLBAM player_id values in exactly the same order as X_model
# and derive bats / throws from Statcast
# ============================================================

import os
import numpy as np
import pandas as pd

DATA_DIR = "../data/raw"
FINAL_DIR = "../data/processed/final"

FEATURE_YEARS = list(range(2015, 2024))
VALID_MONTHS = [4, 5, 6, 7, 8, 9]

BAT_FILE_PATH = os.path.join(DATA_DIR, "war_daily_bat.txt")
PITCH_FILE_PATH = os.path.join(DATA_DIR, "war_daily_pitch.txt")

all_player_ids = []
all_static_rows = []

print("=" * 70)
print("🧍 BUILDING REAL STATIC FEATURES")
print("=" * 70)


def infer_bats(side_values):
    """
    Statcast stand represents the observed batting side:
    - Only R observed -> R
    - Only L observed -> L
    - Both L and R observed in the same season -> switch hitter (S)
    - No data -> U
    """

    sides = {
        str(x).upper()
        for x in side_values
        if pd.notna(x)
    }

    if "S" in sides:
        return "S"

    if "L" in sides and "R" in sides:
        return "S"

    if "L" in sides:
        return "L"

    if "R" in sides:
        return "R"

    return "U"


def infer_throws(hand_values):
    """
    Statcast p_throws represents the pitcher's throwing hand.
    If no pitching data are available, label as U rather than assuming R.
    """

    values = pd.Series(hand_values).dropna().astype(str).str.upper()

    values = values[
        values.isin(["L", "R"])
    ]

    if len(values) == 0:
        return "U"

    return values.mode().iloc[0]


for feature_year in FEATURE_YEARS:

    target_year = feature_year + 1

    raw_path = os.path.join(
        DATA_DIR,
        f"statcast_{feature_year}_full.csv"
    )

    print(
        f"\n📅 {feature_year} Statcast "
        f"→ {target_year} bWAR"
    )

    # --------------------------------------------------------
    # 1. Read only the columns required for static features
    #    Do not rerun the full feature aggregation
    # --------------------------------------------------------

    needed_cols = {
        "game_date",
        "batter",
        "pitcher",
        "stand",
        "p_throws"
    }

    df_static_raw = pd.read_csv(
        raw_path,
        usecols=lambda c: c in needed_cols,
        low_memory=False
    )

    df_static_raw["game_date"] = pd.to_datetime(
        df_static_raw["game_date"]
    )

    df_static_raw["month"] = (
        df_static_raw["game_date"].dt.month
    )

    df_static_raw = df_static_raw[
        df_static_raw["month"].isin(VALID_MONTHS)
    ].copy()

    # --------------------------------------------------------
    # 2. Get WAR players for the target year
    # --------------------------------------------------------

    df_war = get_bwar_and_mapping_local(
        target_year,
        BAT_FILE_PATH,
        PITCH_FILE_PATH
    )

    # --------------------------------------------------------
    # 3. Reproduce the preprocessing valid_players order exactly
    #
    # Original preprocessing:
    # sc_players = df_agg['player_id'].unique()
    # valid_players =
    # np.sort(np.intersect1d(sc_players, war_players))
    # --------------------------------------------------------

    bat_players = (
        df_static_raw["batter"]
        .dropna()
        .astype(np.int64)
        .unique()
    )

    pit_players = (
        df_static_raw["pitcher"]
        .dropna()
        .astype(np.int64)
        .unique()
    )

    sc_players = np.union1d(
        bat_players,
        pit_players
    )

    war_players = (
        df_war["player_id"]
        .dropna()
        .astype(np.int64)
        .unique()
    )

    valid_players = np.sort(
        np.intersect1d(
            sc_players,
            war_players
        )
    )

    # --------------------------------------------------------
    # 4. Build handedness lookup tables in advance
    # --------------------------------------------------------

    bat_side_lookup = (
        df_static_raw
        .dropna(subset=["batter"])
        .groupby("batter")["stand"]
        .apply(infer_bats)
        .to_dict()
    )

    throw_lookup = (
        df_static_raw
        .dropna(subset=["pitcher"])
        .groupby("pitcher")["p_throws"]
        .apply(infer_throws)
        .to_dict()
    )

    # --------------------------------------------------------
    # 5. Build static rows in valid_players order
    # --------------------------------------------------------

    for player_id in valid_players:

        pid = int(player_id)

        bats = bat_side_lookup.get(
            player_id,
            bat_side_lookup.get(pid, "U")
        )

        throws = throw_lookup.get(
            player_id,
            throw_lookup.get(pid, "U")
        )

        all_player_ids.append(pid)

        all_static_rows.append({
            "player_id": pid,
            "feature_year": feature_year,
            "bats": bats,
            "throws": throws
        })

    print(
        f"✅ samples: {len(valid_players)}"
    )


# ============================================================
# 6. Combine and verify exact alignment with the canonical dataset
# ============================================================

model_player_ids = np.asarray(
    all_player_ids,
    dtype=np.int64
)

model_static = pd.DataFrame(
    all_static_rows
)

model_years_check = np.load(
    os.path.join(
        FINAL_DIR,
        "model_years.npy"
    )
)

assert len(model_static) == len(model_years_check)
assert len(model_player_ids) == len(model_years_check)

# Each year's sample count must exactly match the validation dataset
for year in FEATURE_YEARS:

    expected = int(
        np.sum(model_years_check == year)
    )

    actual = int(
        np.sum(
            model_static["feature_year"].values
            == year
        )
    )

    assert expected == actual, (
        f"{year}: expected {expected}, "
        f"but static data has {actual}"
    )


# ============================================================
# 7. Save
# ============================================================

np.save(
    os.path.join(
        FINAL_DIR,
        "model_player_ids.npy"
    ),
    model_player_ids
)

model_static.to_csv(
    os.path.join(
        FINAL_DIR,
        "model_static.csv"
    ),
    index=False
)


# ============================================================
# 8. Diagnostic output
# ============================================================

print("\n" + "=" * 70)
print("✅ REAL STATIC FEATURES COMPLETE")

print(
    f"Total samples : {len(model_static)}"
)

print("\nBats distribution:")
print(
    model_static["bats"]
    .value_counts(dropna=False)
)

print("\nThrows distribution:")
print(
    model_static["throws"]
    .value_counts(dropna=False)
)

print("\nYear distribution:")
print(
    model_static["feature_year"]
    .value_counts()
    .sort_index()
)

print("\nPreview:")
print(model_static.head(10))

print("=" * 70)

## 7. Convert the canonical tensor to TFT long format


In [ ]:
# ============================================================
# FINAL TFT DATA CONVERSION
# Use the canonical dataset + real MLBAM IDs + real handedness
# Do not use synthetic Player_i IDs, age=28, or default R/R values
# ============================================================

import os
import json
import numpy as np
import pandas as pd

FINAL_DIR = "../data/processed/final"

print("=" * 70)
print("🔄 FINAL TFT DATA CONVERSION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Load the canonical model dataset
# ------------------------------------------------------------

X_model = np.load(
    os.path.join(
        FINAL_DIR,
        "X_model.npy"
    )
)

y_model = np.load(
    os.path.join(
        FINAL_DIR,
        "y_model.npy"
    )
)

model_years = np.load(
    os.path.join(
        FINAL_DIR,
        "model_years.npy"
    )
)

model_player_ids = np.load(
    os.path.join(
        FINAL_DIR,
        "model_player_ids.npy"
    )
)

model_static = pd.read_csv(
    os.path.join(
        FINAL_DIR,
        "model_static.csv"
    )
)

with open(
    os.path.join(
        FINAL_DIR,
        "final_feature_names.json"
    ),
    "r"
) as f:
    final_feature_names = json.load(f)


# ------------------------------------------------------------
# 2. Basic integrity checks
# ------------------------------------------------------------

num_samples, num_timesteps, num_features = X_model.shape

assert num_samples == len(y_model)
assert num_samples == len(model_years)
assert num_samples == len(model_player_ids)
assert num_samples == len(model_static)

assert num_timesteps == 6
assert num_features == len(final_feature_names)

assert np.array_equal(
    model_static["feature_year"].values.astype(int),
    model_years.astype(int)
)

assert np.array_equal(
    model_static["player_id"].values.astype(np.int64),
    model_player_ids.astype(np.int64)
)

print(f"Samples    : {num_samples}")
print(f"Time steps : {num_timesteps}")
print(f"Features   : {num_features}")


# ------------------------------------------------------------
# 3. Create genuinely unique TFT time-series IDs
#
# The same MLB player may appear in multiple years,
# so MLBAM ID alone cannot be used as the group ID.
#
# For example:
#   MLBAM 592450 in 2019 -> 592450_2019
#   MLBAM 592450 in 2020 -> 592450_2020
#
# Each player-season is an independent 6-month sequence.
# ------------------------------------------------------------

series_ids = np.array([
    f"{int(pid)}_{int(year)}"
    for pid, year in zip(
        model_player_ids,
        model_years
    )
])

assert len(np.unique(series_ids)) == num_samples


# ------------------------------------------------------------
# 4. Flatten the 3D tensor into a TFT long-format dataframe
# ------------------------------------------------------------

df_meta = pd.DataFrame({
    "series_id": np.repeat(
        series_ids,
        num_timesteps
    ),

    "player_id": np.repeat(
        model_player_ids.astype(np.int64),
        num_timesteps
    ),

    "feature_year": np.repeat(
        model_years.astype(int),
        num_timesteps
    ),

    "time_idx": np.tile(
        np.arange(num_timesteps),
        num_samples
    ),

    "target_war": np.repeat(
        y_model.astype(float),
        num_timesteps
    ),

    "bats": np.repeat(
        model_static["bats"]
        .fillna("U")
        .astype(str)
        .values,
        num_timesteps
    ),

    "throws": np.repeat(
        model_static["throws"]
        .fillna("U")
        .astype(str)
        .values,
        num_timesteps
    )
})


# ------------------------------------------------------------
# 5. Flatten dynamic features
# ------------------------------------------------------------

X_flat = X_model.reshape(
    num_samples * num_timesteps,
    num_features
)

df_features = pd.DataFrame(
    X_flat,
    columns=final_feature_names
)

df_tft = pd.concat(
    [
        df_meta.reset_index(drop=True),
        df_features.reset_index(drop=True)
    ],
    axis=1
)


# ------------------------------------------------------------
# 6. Data type normalization
# ------------------------------------------------------------

df_tft["series_id"] = (
    df_tft["series_id"].astype(str)
)

df_tft["player_id"] = (
    df_tft["player_id"].astype(str)
)

df_tft["bats"] = (
    df_tft["bats"].astype(str)
)

df_tft["throws"] = (
    df_tft["throws"].astype(str)
)

df_tft["time_idx"] = (
    df_tft["time_idx"].astype(int)
)

df_tft["feature_year"] = (
    df_tft["feature_year"].astype(int)
)

df_tft["target_war"] = (
    df_tft["target_war"].astype(float)
)


# ------------------------------------------------------------
# 7. Critical integrity checks
# ------------------------------------------------------------

assert len(df_tft) == num_samples * 6

assert df_tft["series_id"].nunique() == num_samples

assert not df_tft.duplicated(
    subset=["series_id", "time_idx"]
).any()

rows_per_series = (
    df_tft
    .groupby("series_id")
    .size()
)

assert rows_per_series.min() == 6
assert rows_per_series.max() == 6

assert "age" not in df_tft.columns

assert not df_tft[
    ["bats", "throws"]
].isna().any().any()


# ------------------------------------------------------------
# 8. Save the TFT base dataframe
# ------------------------------------------------------------

TFT_PATH = os.path.join(
    FINAL_DIR,
    "tft_base_dataframe.csv"
)

df_tft.to_csv(
    TFT_PATH,
    index=False
)


# ------------------------------------------------------------
# 9. Diagnostic output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ FINAL TFT DATA CONVERSION COMPLETE")
print("=" * 70)

print(
    f"DataFrame shape : {df_tft.shape}"
)

print(
    f"Unique series   : "
    f"{df_tft['series_id'].nunique()}"
)

print(
    f"Unique MLBAM IDs: "
    f"{df_tft['player_id'].nunique()}"
)

print(
    f"Rows per series : "
    f"{rows_per_series.min()}–{rows_per_series.max()}"
)

print("\nFeature-year series count:")

print(
    df_tft[
        ["series_id", "feature_year"]
    ]
    .drop_duplicates()
    ["feature_year"]
    .value_counts()
    .sort_index()
)

print("\nBats distribution (player-season level):")

print(
    model_static["bats"]
    .value_counts(dropna=False)
)

print("\nThrows distribution (player-season level):")

print(
    model_static["throws"]
    .value_counts(dropna=False)
)

print("\nPreview:")

print(
    df_tft[
        [
            "series_id",
            "player_id",
            "feature_year",
            "time_idx",
            "target_war",
            "bats",
            "throws"
        ]
        + final_feature_names[:3]
    ].head(12)
)

print("\n💾 Saved:")
print(TFT_PATH)

print("=" * 70)

## 8. Temporal Fusion Transformer — dissertation checkpoint / optional retraining


In [ ]:
# ============================================================
# FINAL TFT TRAINING PIPELINE
# Canonical features + real static features
# Chronological temporal split
# Train: 2015–2020
# Val  : 2021–2022
# Test : 2023
# ============================================================

import os
import json
import warnings

import numpy as np
import pandas as pd
import torch

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.metrics import RMSE
from pytorch_forecasting.data.encoders import NaNLabelEncoder


warnings.filterwarnings("ignore")

pl.seed_everything(
    42,
    workers=True
)


# ============================================================
# 1. Paths and chronological split
# ============================================================

FINAL_DIR = "../data/processed/final"

TFT_DATA_PATH = os.path.join(
    FINAL_DIR,
    "tft_base_dataframe.csv"
)

FEATURE_PATH = os.path.join(
    FINAL_DIR,
    "final_feature_names.json"
)

TRAIN_YEARS = [
    2015, 2016, 2017,
    2018, 2019, 2020
]

VAL_YEARS = [
    2021, 2022
]

TEST_YEARS = [
    2023
]


print("=" * 70)
print("🚀 FINAL TFT TRAINING PIPELINE")
print("=" * 70)


# ============================================================
# 2. Load the canonical TFT DataFrame created above
# ============================================================

df_tft = pd.read_csv(
    TFT_DATA_PATH,
    dtype={
        "series_id": str,
        "player_id": str,
        "bats": str,
        "throws": str
    }
)

with open(
    FEATURE_PATH,
    "r"
) as f:
    final_feature_names = json.load(f)


print(
    f"Base rows       : {len(df_tft)}"
)

print(
    f"Base series     : "
    f"{df_tft['series_id'].nunique()}"
)

print(
    f"Dynamic features: "
    f"{len(final_feature_names)}"
)


assert len(df_tft) == 57660
assert df_tft["series_id"].nunique() == 9610
assert len(final_feature_names) == 20


# ============================================================
# 3. Handle NaN / Inf values
#
# For role-specific features:
# batter-only / pitcher-only observations may produce NaN values.
#
# To remain consistent with the current zero-padding representation,
# missing dynamic observations are represented as 0.
# ============================================================

df_tft[
    final_feature_names
] = df_tft[
    final_feature_names
].apply(
    pd.to_numeric,
    errors="coerce"
)


bad_counts = {}

for feature in final_feature_names:

    values = df_tft[
        feature
    ].to_numpy(dtype=float)

    bad_count = int(
        (~np.isfinite(values)).sum()
    )

    if bad_count > 0:
        bad_counts[feature] = bad_count


print("\nNon-finite values before cleaning:")

if len(bad_counts) == 0:

    print("✅ None")

else:

    for feature, count in bad_counts.items():

        print(
            f"{feature:40s}: "
            f"{count}"
        )


df_tft[
    final_feature_names
] = (
    df_tft[
        final_feature_names
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0.0)
)


# The target must not be imputed
df_tft["target_war"] = pd.to_numeric(
    df_tft["target_war"],
    errors="coerce"
)

assert np.isfinite(
    df_tft["target_war"]
).all()


# Replace missing static categories with U
df_tft["bats"] = (
    df_tft["bats"]
    .fillna("U")
    .astype(str)
)

df_tft["throws"] = (
    df_tft["throws"]
    .fillna("U")
    .astype(str)
)


# ============================================================
# 4. Create the decoder target step
#
# time_idx:
# 0–5 = April–September encoder
# 6   = next-season WAR decoder target
#
# Future dynamic variables are unavailable at prediction time.
# Therefore, the decoder row does not use any sample-specific future data,
# and instead uses the training-period feature mean as a neutral placeholder.
#
# TimeSeriesDataSet subsequently applies the training scaler,
# so these values are close to 0 after standardization.
# ============================================================

train_base_mask = (
    df_tft["feature_year"]
    .isin(TRAIN_YEARS)
)

train_feature_means = (
    df_tft.loc[
        train_base_mask,
        final_feature_names
    ]
    .mean()
)


# Save for methodology documentation and reproducibility
train_feature_means.to_csv(
    os.path.join(
        FINAL_DIR,
        "tft_train_feature_means.csv"
    ),
    header=["train_mean"]
)


decoder_rows = (
    df_tft[
        df_tft["time_idx"] == 5
    ]
    .copy()
)

decoder_rows[
    "time_idx"
] = 6


for feature in final_feature_names:

    decoder_rows[
        feature
    ] = float(
        train_feature_means[
            feature
        ]
    )


df_tft_extended = pd.concat(
    [
        df_tft,
        decoder_rows
    ],
    ignore_index=True
)


df_tft_extended = (
    df_tft_extended
    .sort_values(
        [
            "series_id",
            "time_idx"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 5. Integrity checks
# ============================================================

rows_per_series = (
    df_tft_extended
    .groupby("series_id")
    .size()
)

assert rows_per_series.min() == 7
assert rows_per_series.max() == 7

assert len(
    df_tft_extended
) == 9610 * 7

assert not df_tft_extended.duplicated(
    subset=[
        "series_id",
        "time_idx"
    ]
).any()


# Recheck all model inputs
assert np.isfinite(
    df_tft_extended[
        final_feature_names
    ].to_numpy(dtype=float)
).all()


print(
    "\n✅ Extended TFT rows:",
    len(df_tft_extended)
)

print(
    "✅ Rows per series:",
    f"{rows_per_series.min()}–"
    f"{rows_per_series.max()}"
)


# ============================================================
# 6. Final chronological Train / Val / Test split
# ============================================================

df_train = (
    df_tft_extended[
        df_tft_extended[
            "feature_year"
        ].isin(TRAIN_YEARS)
    ]
    .copy()
)

df_val = (
    df_tft_extended[
        df_tft_extended[
            "feature_year"
        ].isin(VAL_YEARS)
    ]
    .copy()
)

df_test = (
    df_tft_extended[
        df_tft_extended[
            "feature_year"
        ].isin(TEST_YEARS)
    ]
    .copy()
)


n_train_series = (
    df_train["series_id"]
    .nunique()
)

n_val_series = (
    df_val["series_id"]
    .nunique()
)

n_test_series = (
    df_test["series_id"]
    .nunique()
)


assert n_train_series == 6293
assert n_val_series == 2211
assert n_test_series == 1106


print("\n" + "=" * 70)
print("CHRONOLOGICAL TFT SPLIT")
print("=" * 70)

print(
    f"Train: {n_train_series} series "
    f"| {len(df_train)} rows "
    f"| years {TRAIN_YEARS}"
)

print(
    f"Val  : {n_val_series} series "
    f"| {len(df_val)} rows "
    f"| years {VAL_YEARS}"
)

print(
    f"Test : {n_test_series} series "
    f"| {len(df_test)} rows "
    f"| years {TEST_YEARS}"
)


# ============================================================
# 7. Build TimeSeriesDataSet
#
# Note:
# Do not include target_war in time_varying_unknown_reals.
#
# target_war is the subsequent-season WAR target,
# not a historical monthly target observed during the six encoder months.
# Including it would cause target leakage.
# ============================================================

print("\n📦 Building TimeSeriesDataSet...")


training_dataset = TimeSeriesDataSet(

    df_train,

    time_idx="time_idx",

    target="target_war",

    group_ids=[
        "series_id"
    ],

    min_encoder_length=6,
    max_encoder_length=6,

    min_prediction_length=1,
    max_prediction_length=1,

    static_categoricals=[
        "bats",
        "throws"
    ],

    time_varying_known_reals=[
        "time_idx"
    ],

    time_varying_unknown_reals=(
        final_feature_names
    ),

    categorical_encoders={
        "bats": NaNLabelEncoder(
            add_nan=True
        ),
        "throws": NaNLabelEncoder(
            add_nan=True
        )
    },

    target_normalizer=None,

    add_relative_time_idx=True,
    add_target_scales=False,
    add_encoder_length=False,

    allow_missing_timesteps=False
)


# Validation / Test reuse the training dataset's
# learned encoding and scaling
validation_dataset = (
    TimeSeriesDataSet.from_dataset(

        training_dataset,

        df_val,

        predict=True,

        stop_randomization=True
    )
)


test_dataset = (
    TimeSeriesDataSet.from_dataset(

        training_dataset,

        df_test,

        predict=True,

        stop_randomization=True
    )
)


print(
    "Training dataset:",
    len(training_dataset)
)

print(
    "Validation dataset:",
    len(validation_dataset)
)

print(
    "Test dataset:",
    len(test_dataset)
)


assert len(training_dataset) == 6293
assert len(validation_dataset) == 2211
assert len(test_dataset) == 1106


# ============================================================
# 8. DataLoaders
# ============================================================

batch_size = 64


train_dataloader = (
    training_dataset.to_dataloader(

        train=True,

        batch_size=batch_size,

        num_workers=0
    )
)


val_dataloader = (
    validation_dataset.to_dataloader(

        train=False,

        batch_size=batch_size,

        num_workers=0
    )
)


test_dataloader = (
    test_dataset.to_dataloader(

        train=False,

        batch_size=batch_size,

        num_workers=0
    )
)


# ============================================================
# 9. Dissertation checkpoint / optional retraining
#
# Default:
#   RETRAIN_TFT = False
#   -> load the exact checkpoint used for the dissertation.
#
# Set RETRAIN_TFT = True only when intentionally training
# a new TFT model. A retrained model is saved separately and
# does NOT overwrite the dissertation checkpoint.
# ============================================================

RETRAIN_TFT = False

DISSERTATION_CHECKPOINT = (
    "../models/tft_dissertation.ckpt"
)


if RETRAIN_TFT:

    print("\n🔁 Training a new TFT model...")

    tft_model = (
        TemporalFusionTransformer
        .from_dataset(
            training_dataset,
            learning_rate=0.01,
            hidden_size=32,
            attention_head_size=4,
            dropout=0.1,
            hidden_continuous_size=16,
            loss=RMSE(),
            log_interval=10,
            optimizer="Adam",
            reduce_on_plateau_patience=4
        )
    )

    early_stop_callback = EarlyStopping(
        monitor="val_loss",
        min_delta=1e-4,
        patience=5,
        verbose=True,
        mode="min"
    )

    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        dirpath=FINAL_DIR,
        filename="tft_retrained",
        save_top_k=1,
        mode="min"
    )

    trainer = pl.Trainer(
        max_epochs=30,
        accelerator="auto",
        callbacks=[
            early_stop_callback,
            checkpoint_callback
        ],
        enable_model_summary=True
    )

    trainer.fit(
        tft_model,
        train_dataloaders=train_dataloader,
        val_dataloaders=val_dataloader
    )

    best_model_path = (
        checkpoint_callback.best_model_path
    )

else:

    print(
        "\n📦 Loading dissertation TFT checkpoint..."
    )

    best_model_path = (
        DISSERTATION_CHECKPOINT
    )


print("\n🏆 TFT checkpoint:")
print(best_model_path)


best_tft = (
    TemporalFusionTransformer
    .load_from_checkpoint(
        best_model_path,
        weights_only=False
    )
)

print("✅ TFT model ready.")


## 9. Train the LSTM


In [ ]:
# ============================================================
# FINAL LSTM TRAINING PIPELINE
# Same canonical features + same chronological split as TFT
#
# Train: 2015–2020
# Val  : 2021–2022
# Test : 2023 -> 2024 bWAR
# ============================================================

import os
import json
import copy
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. Paths
# ============================================================

FINAL_DIR = "../data/processed/final"

X_PATH = os.path.join(
    FINAL_DIR,
    "X_model.npy"
)

Y_PATH = os.path.join(
    FINAL_DIR,
    "y_model.npy"
)

YEARS_PATH = os.path.join(
    FINAL_DIR,
    "model_years.npy"
)

FEATURE_PATH = os.path.join(
    FINAL_DIR,
    "final_feature_names.json"
)


print("=" * 70)
print("🚀 FINAL LSTM TRAINING PIPELINE")
print("=" * 70)


# ============================================================
# 3. Load canonical dataset
# ============================================================

X = np.load(X_PATH).astype(np.float32)
y = np.load(Y_PATH).astype(np.float32)
years = np.load(YEARS_PATH).astype(int)

with open(
    FEATURE_PATH,
    "r"
) as f:
    feature_names = json.load(f)


assert X.shape == (9610, 6, 20)
assert len(y) == 9610
assert len(years) == 9610
assert len(feature_names) == 20


print(f"Full X shape : {X.shape}")
print(f"Full y shape : {y.shape}")
print(f"Features     : {len(feature_names)}")


# ============================================================
# 4. Chronological split
# ============================================================

TRAIN_YEARS = [
    2015, 2016, 2017,
    2018, 2019, 2020
]

VAL_YEARS = [
    2021, 2022
]

TEST_YEARS = [
    2023
]


train_mask = np.isin(
    years,
    TRAIN_YEARS
)

val_mask = np.isin(
    years,
    VAL_YEARS
)

test_mask = np.isin(
    years,
    TEST_YEARS
)


X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

X_test = X[test_mask]
y_test = y[test_mask]


assert len(X_train) == 6293
assert len(X_val) == 2211
assert len(X_test) == 1106


print("\n" + "=" * 70)
print("CHRONOLOGICAL LSTM SPLIT")
print("=" * 70)

print(
    f"Train : {X_train.shape} "
    f"| years {TRAIN_YEARS}"
)

print(
    f"Val   : {X_val.shape} "
    f"| years {VAL_YEARS}"
)

print(
    f"Test  : {X_test.shape} "
    f"| years {TEST_YEARS}"
)


# ============================================================
# 5. Check NaN / Inf
# ============================================================

for name, arr in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test),
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)
]:

    bad = (
        ~np.isfinite(arr)
    ).sum()

    print(
        f"{name:10s} non-finite: "
        f"{int(bad)}"
    )

    assert bad == 0


# ============================================================
# 6. Train-only feature standardization
#
# Scaler is fitted ONLY using training observations.
#
# Shape for fitting:
# (6293 × 6, 20)
# ============================================================

scaler = StandardScaler()

X_train_2d = X_train.reshape(
    -1,
    X_train.shape[-1]
)

scaler.fit(
    X_train_2d
)


def transform_3d(X_input, scaler):

    original_shape = (
        X_input.shape
    )

    X_flat = X_input.reshape(
        -1,
        original_shape[-1]
    )

    X_scaled = scaler.transform(
        X_flat
    )

    return (
        X_scaled
        .reshape(original_shape)
        .astype(np.float32)
    )


X_train_scaled = transform_3d(
    X_train,
    scaler
)

X_val_scaled = transform_3d(
    X_val,
    scaler
)

X_test_scaled = transform_3d(
    X_test,
    scaler
)


print(
    "\n✅ StandardScaler fitted "
    "using TRAIN data only."
)


# ============================================================
# 7. Dataset
# ============================================================

class MLBSequenceDataset(Dataset):

    def __init__(
        self,
        X,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        ).view(
            -1,
            1
        )

    def __len__(self):

        return len(
            self.y
        )

    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )


train_dataset = MLBSequenceDataset(
    X_train_scaled,
    y_train
)

val_dataset = MLBSequenceDataset(
    X_val_scaled,
    y_val
)

test_dataset = MLBSequenceDataset(
    X_test_scaled,
    y_test
)


BATCH_SIZE = 64


generator = torch.Generator()
generator.manual_seed(SEED)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ============================================================
# 8. Baseline LSTM
#
# Preserve original main architecture:
# hidden_dim = 32
# num_layers = 1
# ============================================================

class BaselineLSTM(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        num_layers=1
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            1
        )

    def forward(
        self,
        x
    ):

        lstm_out, _ = self.lstm(
            x
        )

        last_hidden = lstm_out[
            :,
            -1,
            :
        ]

        output = self.fc(
            last_hidden
        )

        return output


# ============================================================
# 9. Device
# ============================================================

if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif (
    hasattr(torch.backends, "mps")
    and
    torch.backends.mps.is_available()
):

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )


print(
    f"\nDevice: {device}"
)


# ============================================================
# 10. Model / loss / optimizer
#
# Keep original LSTM settings:
# epochs = 30
# lr = 0.005
# hidden_dim = 32
# batch_size = 64
# MSE + Adam
# ============================================================

INPUT_DIM = X_train_scaled.shape[-1]
HIDDEN_DIM = 32
NUM_LAYERS = 1

MAX_EPOCHS = 30
LEARNING_RATE = 0.005


lstm_model = BaselineLSTM(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS
).to(
    device
)


criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=LEARNING_RATE
)


total_params = sum(
    p.numel()
    for p in lstm_model.parameters()
)


print(
    f"LSTM parameters: "
    f"{total_params:,}"
)


# ============================================================
# 11. Training
#
# Train for fixed 30 epochs.
# Best checkpoint chosen ONLY using validation loss.
# Test set is never used during training.
# ============================================================

best_val_loss = np.inf
best_epoch = None
best_state = None

history = []


print(
    "\n🔥 Training FINAL LSTM..."
)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # -------------------------
    # Train
    # -------------------------

    lstm_model.train()

    train_loss_sum = 0.0
    train_n = 0


    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(
            device
        )

        batch_y = batch_y.to(
            device
        )

        optimizer.zero_grad()

        predictions = lstm_model(
            batch_X
        )

        loss = criterion(
            predictions,
            batch_y
        )

        loss.backward()

        optimizer.step()

        batch_size_actual = (
            batch_X.size(0)
        )

        train_loss_sum += (
            loss.item()
            * batch_size_actual
        )

        train_n += (
            batch_size_actual
        )


    train_mse = (
        train_loss_sum
        / train_n
    )


    # -------------------------
    # Validation
    # -------------------------

    lstm_model.eval()

    val_loss_sum = 0.0
    val_n = 0


    with torch.no_grad():

        for batch_X, batch_y in val_loader:

            batch_X = batch_X.to(
                device
            )

            batch_y = batch_y.to(
                device
            )

            predictions = lstm_model(
                batch_X
            )

            loss = criterion(
                predictions,
                batch_y
            )

            batch_size_actual = (
                batch_X.size(0)
            )

            val_loss_sum += (
                loss.item()
                * batch_size_actual
            )

            val_n += (
                batch_size_actual
            )


    val_mse = (
        val_loss_sum
        / val_n
    )

    train_rmse = np.sqrt(
        train_mse
    )

    val_rmse = np.sqrt(
        val_mse
    )


    history.append({
        "epoch": epoch,
        "train_mse": train_mse,
        "train_rmse": train_rmse,
        "val_mse": val_mse,
        "val_rmse": val_rmse
    })


    # -------------------------
    # Save best validation state
    # -------------------------

    if val_mse < best_val_loss:

        best_val_loss = val_mse
        best_epoch = epoch

        best_state = copy.deepcopy(
            lstm_model.state_dict()
        )


    print(
        f"Epoch {epoch:02d}/"
        f"{MAX_EPOCHS} "
        f"| Train RMSE: "
        f"{train_rmse:.4f} "
        f"| Val RMSE: "
        f"{val_rmse:.4f}"
    )


# ============================================================
# 12. Restore best validation checkpoint
# ============================================================

assert best_state is not None

lstm_model.load_state_dict(
    best_state
)


print(
    "\n🏆 Best validation epoch:",
    best_epoch
)

print(
    "🏆 Best validation RMSE:",
    f"{np.sqrt(best_val_loss):.4f}"
)


# ============================================================
# 13. Final TEST prediction
# ============================================================

lstm_model.eval()

test_predictions = []


with torch.no_grad():

    for batch_X, _ in test_loader:

        batch_X = batch_X.to(
            device
        )

        preds = lstm_model(
            batch_X
        )

        test_predictions.extend(
            preds
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
            .tolist()
        )


test_predictions = np.array(
    test_predictions
)


assert len(
    test_predictions
) == len(
    y_test
)


# ============================================================
# 14. Final metrics
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_r2 = r2_score(
    y_test,
    test_predictions
)


print(
    "\n" + "=" * 70
)

print(
    "🎯 FINAL LSTM TEST RESULTS"
)

print(
    "=" * 70
)

print(
    f"RMSE : {test_rmse:.6f}"
)

print(
    f"MAE  : {test_mae:.6f}"
)

print(
    f"R²   : {test_r2:.6f}"
)

print(
    f"Best validation epoch: "
    f"{best_epoch}"
)

print(
    "=" * 70
)


# ============================================================
# 15. Save final model
# ============================================================

MODEL_PATH = os.path.join(
    FINAL_DIR,
    "lstm_best_temporal.pt"
)

torch.save(
    {
        "model_state_dict":
            lstm_model.state_dict(),

        "input_dim":
            INPUT_DIM,

        "hidden_dim":
            HIDDEN_DIM,

        "num_layers":
            NUM_LAYERS,

        "best_epoch":
            best_epoch,

        "feature_names":
            feature_names
    },
    MODEL_PATH
)


# ============================================================
# 16. Save scaler
# ============================================================

SCALER_PATH = os.path.join(
    FINAL_DIR,
    "lstm_scaler.npz"
)

np.savez(
    SCALER_PATH,
    mean=scaler.mean_,
    scale=scaler.scale_,
    var=scaler.var_
)


# ============================================================
# 17. Save predictions
# ============================================================

PRED_PATH = os.path.join(
    FINAL_DIR,
    "lstm_test_predictions.csv"
)

pd.DataFrame({

    "feature_year":
        years[test_mask],

    "target_year":
        years[test_mask] + 1,

    "actual_bwar":
        y_test,

    "predicted_bwar":
        test_predictions

}).to_csv(
    PRED_PATH,
    index=False
)


# ============================================================
# 18. Save training history
# ============================================================

HISTORY_PATH = os.path.join(
    FINAL_DIR,
    "lstm_training_history.csv"
)

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False
)


# ============================================================
# 19. Save metrics
# ============================================================

METRICS_PATH = os.path.join(
    FINAL_DIR,
    "lstm_test_metrics.json"
)

metrics = {

    "rmse":
        float(test_rmse),

    "mae":
        float(test_mae),

    "r2":
        float(test_r2),

    "best_validation_epoch":
        int(best_epoch),

    "best_validation_rmse":
        float(
            np.sqrt(
                best_val_loss
            )
        ),

    "train_samples":
        int(len(X_train)),

    "validation_samples":
        int(len(X_val)),

    "test_samples":
        int(len(X_test))
}


with open(
    METRICS_PATH,
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=2
    )


print("\n💾 Saved:")
print(MODEL_PATH)
print(SCALER_PATH)
print(PRED_PATH)
print(HISTORY_PATH)
print(METRICS_PATH)

## 10. Train Ridge Regression and Random Forest baselines


In [ ]:
# ============================================================
# FINAL RIDGE + RANDOM FOREST BASELINES
#
# Same canonical 20 features
# Same chronological temporal split
#
# Train: 2015–2020
# Val  : 2021–2022
# Test : 2023 -> 2024 bWAR
# ============================================================

import os
import json

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. Paths
# ============================================================

FINAL_DIR = "../data/processed/final"

X_PATH = os.path.join(
    FINAL_DIR,
    "X_model.npy"
)

Y_PATH = os.path.join(
    FINAL_DIR,
    "y_model.npy"
)

YEARS_PATH = os.path.join(
    FINAL_DIR,
    "model_years.npy"
)

FEATURE_PATH = os.path.join(
    FINAL_DIR,
    "final_feature_names.json"
)


print("=" * 70)
print("🚀 FINAL RIDGE + RANDOM FOREST BASELINES")
print("=" * 70)


# ============================================================
# 2. Load canonical dataset
# ============================================================

X = np.load(
    X_PATH
).astype(
    np.float32
)

y = np.load(
    Y_PATH
).astype(
    np.float32
)

years = np.load(
    YEARS_PATH
).astype(
    int
)


with open(
    FEATURE_PATH,
    "r"
) as f:

    feature_names = json.load(f)


assert X.shape == (
    9610,
    6,
    20
)

assert len(y) == 9610
assert len(years) == 9610
assert len(feature_names) == 20


print(
    f"Full dataset : {X.shape}"
)

print(
    f"Features     : {len(feature_names)}"
)


# ============================================================
# 3. Same chronological split as LSTM / TFT
# ============================================================

TRAIN_YEARS = [
    2015, 2016, 2017,
    2018, 2019, 2020
]

VAL_YEARS = [
    2021, 2022
]

TEST_YEARS = [
    2023
]


train_mask = np.isin(
    years,
    TRAIN_YEARS
)

val_mask = np.isin(
    years,
    VAL_YEARS
)

test_mask = np.isin(
    years,
    TEST_YEARS
)


X_train = X[
    train_mask
]

y_train = y[
    train_mask
]

X_val = X[
    val_mask
]

y_val = y[
    val_mask
]

X_test = X[
    test_mask
]

y_test = y[
    test_mask
]


assert len(X_train) == 6293
assert len(X_val) == 2211
assert len(X_test) == 1106


print("\n" + "=" * 70)
print("CHRONOLOGICAL BASELINE SPLIT")
print("=" * 70)

print(
    f"Train : {X_train.shape}"
)

print(
    f"Val   : {X_val.shape}"
)

print(
    f"Test  : {X_test.shape}"
)


# ============================================================
# 4. Check NaN / Inf
# ============================================================

for name, arr in [

    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test),

    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)

]:

    bad = int(
        (~np.isfinite(arr))
        .sum()
    )

    print(
        f"{name:10s} non-finite: "
        f"{bad}"
    )

    assert bad == 0


# ============================================================
# 5. Convert monthly sequence to season-level representation
#
# Ridge and Random Forest are non-sequential baselines.
# Therefore, the six monthly observations are aggregated
# into one season-level value per feature.
#
# Shape:
# (N, 6, 20) -> (N, 20)
# ============================================================

X_train_2d = X_train.mean(
    axis=1
)

X_val_2d = X_val.mean(
    axis=1
)

X_test_2d = X_test.mean(
    axis=1
)


assert X_train_2d.shape == (
    6293,
    20
)

assert X_val_2d.shape == (
    2211,
    20
)

assert X_test_2d.shape == (
    1106,
    20
)


print("\n✅ Season-level aggregation complete")

print(
    "Train:",
    X_train_2d.shape
)

print(
    "Val  :",
    X_val_2d.shape
)

print(
    "Test :",
    X_test_2d.shape
)


# ============================================================
# 6. Train-only standardization for Ridge
#
# Ridge regularization is scale-sensitive.
# StandardScaler is therefore fitted ONLY on training data.
#
# Random Forest does NOT require standardization.
# ============================================================

ridge_scaler = StandardScaler()

X_train_ridge = (
    ridge_scaler
    .fit_transform(
        X_train_2d
    )
)

X_val_ridge = (
    ridge_scaler
    .transform(
        X_val_2d
    )
)

X_test_ridge = (
    ridge_scaler
    .transform(
        X_test_2d
    )
)


print(
    "\n✅ Ridge scaler fitted "
    "using TRAIN data only."
)


# ============================================================
# 7. Ridge Regression
#
# alpha = 1.0 retained from original baseline
#
# Ridge itself is deterministic.
# ============================================================

print(
    "\n📈 Training Ridge Regression..."
)


ridge_model = Ridge(
    alpha=1.0
)


ridge_model.fit(
    X_train_ridge,
    y_train
)


ridge_val_preds = (
    ridge_model.predict(
        X_val_ridge
    )
)

ridge_test_preds = (
    ridge_model.predict(
        X_test_ridge
    )
)


# ============================================================
# 8. Random Forest
#
# Retain original:
# n_estimators = 100
# random_state = 42
#
# n_jobs = 1 used for stronger run-to-run reproducibility.
# ============================================================

print(
    "🌲 Training Random Forest..."
)


rf_model = RandomForestRegressor(

    n_estimators=100,

    random_state=42,

    n_jobs=1
)


rf_model.fit(
    X_train_2d,
    y_train
)


rf_val_preds = (
    rf_model.predict(
        X_val_2d
    )
)

rf_test_preds = (
    rf_model.predict(
        X_test_2d
    )
)


# ============================================================
# 9. Evaluation helper
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2)
    }


# ============================================================
# 10. Validation metrics
# ============================================================

ridge_val_metrics = (
    calculate_metrics(
        y_val,
        ridge_val_preds
    )
)

rf_val_metrics = (
    calculate_metrics(
        y_val,
        rf_val_preds
    )
)


print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(
    "Ridge Regression"
)

print(
    f"RMSE : "
    f"{ridge_val_metrics['rmse']:.6f}"
)

print(
    f"MAE  : "
    f"{ridge_val_metrics['mae']:.6f}"
)

print(
    f"R²   : "
    f"{ridge_val_metrics['r2']:.6f}"
)


print("\nRandom Forest")

print(
    f"RMSE : "
    f"{rf_val_metrics['rmse']:.6f}"
)

print(
    f"MAE  : "
    f"{rf_val_metrics['mae']:.6f}"
)

print(
    f"R²   : "
    f"{rf_val_metrics['r2']:.6f}"
)


# ============================================================
# 11. FINAL TEST metrics
# ============================================================

ridge_test_metrics = (
    calculate_metrics(
        y_test,
        ridge_test_preds
    )
)

rf_test_metrics = (
    calculate_metrics(
        y_test,
        rf_test_preds
    )
)


print("\n" + "=" * 70)
print("🎯 FINAL BASELINE TEST RESULTS")
print("=" * 70)


print("\nRidge Regression")

print(
    f"RMSE : "
    f"{ridge_test_metrics['rmse']:.6f}"
)

print(
    f"MAE  : "
    f"{ridge_test_metrics['mae']:.6f}"
)

print(
    f"R²   : "
    f"{ridge_test_metrics['r2']:.6f}"
)


print("\nRandom Forest")

print(
    f"RMSE : "
    f"{rf_test_metrics['rmse']:.6f}"
)

print(
    f"MAE  : "
    f"{rf_test_metrics['mae']:.6f}"
)

print(
    f"R²   : "
    f"{rf_test_metrics['r2']:.6f}"
)


print("=" * 70)


# ============================================================
# 12. Save predictions
# ============================================================

prediction_df = pd.DataFrame({

    "feature_year":
        years[test_mask],

    "target_year":
        years[test_mask] + 1,

    "actual_bwar":
        y_test,

    "ridge_predicted_bwar":
        ridge_test_preds,

    "rf_predicted_bwar":
        rf_test_preds

})


PRED_PATH = os.path.join(

    FINAL_DIR,

    "baseline_test_predictions.csv"
)


prediction_df.to_csv(
    PRED_PATH,
    index=False
)


# ============================================================
# 13. Save metrics
# ============================================================

metrics = {

    "ridge": {

        "validation":
            ridge_val_metrics,

        "test":
            ridge_test_metrics,

        "alpha":
            1.0
    },

    "random_forest": {

        "validation":
            rf_val_metrics,

        "test":
            rf_test_metrics,

        "n_estimators":
            100,

        "random_state":
            42
    },

    "train_samples":
        int(len(X_train)),

    "validation_samples":
        int(len(X_val)),

    "test_samples":
        int(len(X_test))
}


METRICS_PATH = os.path.join(

    FINAL_DIR,

    "baseline_test_metrics.json"
)


with open(
    METRICS_PATH,
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=2
    )


# ============================================================
# 14. Save Ridge scaler
# ============================================================

SCALER_PATH = os.path.join(

    FINAL_DIR,

    "ridge_scaler.npz"
)


np.savez(

    SCALER_PATH,

    mean=ridge_scaler.mean_,

    scale=ridge_scaler.scale_,

    var=ridge_scaler.var_
)


print("\n💾 Saved:")

print(PRED_PATH)
print(METRICS_PATH)
print(SCALER_PATH)

## 11. LSTM SHAP interpretability


In [ ]:
# ============================================================
# FINAL LSTM SHAP INTERPRETABILITY
#
# Uses:
# - FINAL LSTM checkpoint already selected by validation set
# - canonical 20 features
# - chronological 2024 test set
# - training-only StandardScaler
#
# Outputs:
# 1. Global SHAP importance
# 2. Batter SHAP beeswarm
# 3. Pitcher SHAP beeswarm
# ============================================================

import os
import copy
import json

import numpy as np
import pandas as pd
import torch
import shap
import matplotlib.pyplot as plt


print("=" * 70)
print("🔍 FINAL LSTM SHAP INTERPRETABILITY")
print("=" * 70)


# ============================================================
# 1. Paths
# ============================================================

FINAL_DIR = "../data/processed/final"
VIS_DIR = "../data/visualizations"

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

os.makedirs(
    VIS_DIR,
    exist_ok=True
)


# ============================================================
# 2. Basic checks
# ============================================================

assert X_train_scaled.shape == (
    6293,
    6,
    20
)

assert X_test_scaled.shape == (
    1106,
    6,
    20
)

assert X_test.shape == (
    1106,
    6,
    20
)

assert len(feature_names) == 20


print(
    "Train shape:",
    X_train_scaled.shape
)

print(
    "Test shape :",
    X_test_scaled.shape
)

print(
    "Features   :",
    len(feature_names)
)


# ============================================================
# 3. Move a COPY of final LSTM to CPU for SHAP
#
# This avoids possible MPS / SHAP compatibility problems.
# The trained model itself is not modified.
# ============================================================

shap_model = copy.deepcopy(
    lstm_model
)

shap_model = shap_model.to(
    "cpu"
)

shap_model.eval()


print(
    "✅ Final LSTM copied to CPU for SHAP."
)


# ============================================================
# 4. Deterministic SHAP sampling
#
# Fixed seed = 42
#
# Background:
# 200 TRAIN samples
#
# Explanation:
# 200 TEST samples
# ============================================================

SHAP_SEED = 42

rng = np.random.default_rng(
    SHAP_SEED
)


BACKGROUND_N = min(
    200,
    len(X_train_scaled)
)

EXPLAIN_N = min(
    200,
    len(X_test_scaled)
)


background_idx = rng.choice(
    len(X_train_scaled),
    size=BACKGROUND_N,
    replace=False
)


explain_idx = rng.choice(
    len(X_test_scaled),
    size=EXPLAIN_N,
    replace=False
)


background_np = (
    X_train_scaled[
        background_idx
    ]
    .astype(
        np.float32
    )
)


explain_scaled_np = (
    X_test_scaled[
        explain_idx
    ]
    .astype(
        np.float32
    )
)


# Original unscaled values are retained
# only for role identification and plotting.
explain_raw_np = (
    X_test[
        explain_idx
    ]
    .astype(
        np.float32
    )
)


background_tensor = torch.tensor(
    background_np,
    dtype=torch.float32
)

explain_tensor = torch.tensor(
    explain_scaled_np,
    dtype=torch.float32
)


print(
    f"Background samples : {BACKGROUND_N}"
)

print(
    f"Explained samples  : {EXPLAIN_N}"
)


# ============================================================
# 5. GradientExplainer
# ============================================================

print(
    "\n Computing Gradient SHAP values..."
)


# Fix SHAP's internal stochastic sampling
np.random.seed(SHAP_SEED)
torch.manual_seed(SHAP_SEED)


explainer = shap.GradientExplainer(
    shap_model,
    background_tensor
)


shap_values = explainer.shap_values(
    explain_tensor
)


# PyTorch / SHAP versions may return
# either ndarray or a one-element list.
if isinstance(
    shap_values,
    list
):

    shap_values = (
        shap_values[0]
    )


shap_values = np.asarray(
    shap_values
)


# Typical single-output shape:
# (N, 6, 20, 1)
if (
    shap_values.ndim == 4
    and
    shap_values.shape[-1] == 1
):

    shap_values = (
        shap_values
        .squeeze(-1)
    )


print(
    "Raw SHAP shape:",
    shap_values.shape
)


assert shap_values.shape == (
    EXPLAIN_N,
    6,
    20
), (
    "Unexpected SHAP shape: "
    f"{shap_values.shape}"
)


# ============================================================
# 6. Temporal aggregation
#
# Signed SHAP contributions are summed across
# April–September to obtain one contribution
# per feature for each player-season.
#
# Feature values are averaged across months
# for visualization.
# ============================================================

shap_aggregated = np.sum(
    shap_values,
    axis=1
)


feature_values_aggregated = np.mean(
    explain_raw_np,
    axis=1
)


assert shap_aggregated.shape == (
    EXPLAIN_N,
    20
)


print(
    "Aggregated SHAP shape:",
    shap_aggregated.shape
)


# ============================================================
# 7. GLOBAL SHAP IMPORTANCE
#
# Mean absolute SHAP value across explained test samples.
# ============================================================

global_mean_abs_shap = np.mean(
    np.abs(
        shap_aggregated
    ),
    axis=0
)


global_importance = pd.DataFrame({

    "Feature":
        feature_names,

    "Mean_Abs_SHAP":
        global_mean_abs_shap

})


global_importance = (
    global_importance
    .sort_values(
        "Mean_Abs_SHAP",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


global_total = (
    global_importance[
        "Mean_Abs_SHAP"
    ].sum()
)


global_importance[
    "Importance_Percent"
] = (

    global_importance[
        "Mean_Abs_SHAP"
    ]

    / global_total

    * 100
)


print(
    "\n" + "=" * 70
)

print(
    "GLOBAL LSTM SHAP IMPORTANCE"
)

print(
    "=" * 70
)

print(
    global_importance
    .to_string(
        index=False
    )
)


GLOBAL_CSV = os.path.join(
    FINAL_DIR,
    "lstm_shap_global_importance.csv"
)


global_importance.to_csv(
    GLOBAL_CSV,
    index=False
)


# ============================================================
# 8. Global SHAP bar plot
# ============================================================

TOP_N = min(
    15,
    len(
        global_importance
    )
)


global_plot_df = (
    global_importance
    .head(
        TOP_N
    )
    .sort_values(
        "Importance_Percent",
        ascending=True
    )
)


plt.figure(
    figsize=(10, 7)
)


plt.barh(
    global_plot_df[
        "Feature"
    ],
    global_plot_df[
        "Importance_Percent"
    ]
)


plt.xlabel(
    "Mean Absolute SHAP Importance (%)"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "LSTM Global SHAP Feature Importance"
)

plt.tight_layout()


GLOBAL_PNG = os.path.join(
    VIS_DIR,
    "lstm_shap_global_importance.png"
)


plt.savefig(
    GLOBAL_PNG,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. Identify batter / pitcher features
# ============================================================

bat_indices = [

    i

    for i, name
    in enumerate(
        feature_names
    )

    if name.startswith(
        "bat_"
    )
]


pit_indices = [

    i

    for i, name
    in enumerate(
        feature_names
    )

    if name.startswith(
        "pit_"
    )
]


print(
    "\nBatter features:",
    len(bat_indices)
)

print(
    "Pitcher features:",
    len(pit_indices)
)


# ============================================================
# 10. Identify player-season roles
#
# A player is classified as having batting/pitching
# activity if ANY corresponding raw feature is non-zero
# across the six monthly observations.
#
# This avoids excluding batters with zero hits or
# pitchers with zero strikeouts.
#
# Two-way players may appear in BOTH groups.
# ============================================================

ROLE_EPS = 1e-12

bat_activity_matrix = np.abs(
    explain_raw_np[:, :, bat_indices]
)

pit_activity_matrix = np.abs(
    explain_raw_np[:, :, pit_indices]
)

is_batter = np.any(
    bat_activity_matrix > ROLE_EPS,
    axis=(1, 2)
)

is_pitcher = np.any(
    pit_activity_matrix > ROLE_EPS,
    axis=(1, 2)
)

print(
    f"\nBatter SHAP samples : "
    f"{int(is_batter.sum())}"
)

print(
    f"Pitcher SHAP samples: "
    f"{int(is_pitcher.sum())}"
)


# ============================================================
# 11. BATTER SHAP
# ============================================================

if (
    is_batter.any()
    and
    len(bat_indices) > 0
):

    shap_bat = (
        shap_aggregated[
            is_batter
        ][:, bat_indices]
    )


    feat_bat = (
        feature_values_aggregated[
            is_batter
        ][:, bat_indices]
    )


    names_bat = [

        feature_names[i]

        for i
        in bat_indices
    ]


    batter_mean_abs = np.mean(
        np.abs(
            shap_bat
        ),
        axis=0
    )


    batter_df = pd.DataFrame({

        "Feature":
            names_bat,

        "Mean_Abs_SHAP":
            batter_mean_abs

    }).sort_values(

        "Mean_Abs_SHAP",
        ascending=False

    ).reset_index(
        drop=True
    )


    batter_df[
        "Importance_Percent"
    ] = (

        batter_df[
            "Mean_Abs_SHAP"
        ]

        / batter_df[
            "Mean_Abs_SHAP"
        ].sum()

        * 100
    )


    print(
        "\n" + "=" * 70
    )

    print(
        "BATTER SHAP IMPORTANCE"
    )

    print(
        "=" * 70
    )

    print(
        batter_df
        .to_string(
            index=False
        )
    )


    BATTER_CSV = os.path.join(
        FINAL_DIR,
        "lstm_shap_batter_importance.csv"
    )


    batter_df.to_csv(
        BATTER_CSV,
        index=False
    )


    plt.figure(
        figsize=(10, 7)
    )


    shap.summary_plot(
        shap_bat,
        features=feat_bat,
        feature_names=names_bat,
        max_display=min(
            12,
            len(names_bat)
        ),
        show=False
    )


    plt.title(
        "LSTM SHAP Summary – Batter Features"
    )

    plt.tight_layout()


    BATTER_PNG = os.path.join(
        VIS_DIR,
        "lstm_shap_batters.png"
    )


    plt.savefig(
        BATTER_PNG,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


else:

    BATTER_CSV = None
    BATTER_PNG = None

    print(
        "⚠️ No batter samples found."
    )


# ============================================================
# 12. PITCHER SHAP
# ============================================================

if (
    is_pitcher.any()
    and
    len(pit_indices) > 0
):

    shap_pit = (
        shap_aggregated[
            is_pitcher
        ][:, pit_indices]
    )


    feat_pit = (
        feature_values_aggregated[
            is_pitcher
        ][:, pit_indices]
    )


    names_pit = [

        feature_names[i]

        for i
        in pit_indices
    ]


    pitcher_mean_abs = np.mean(
        np.abs(
            shap_pit
        ),
        axis=0
    )


    pitcher_df = pd.DataFrame({

        "Feature":
            names_pit,

        "Mean_Abs_SHAP":
            pitcher_mean_abs

    }).sort_values(

        "Mean_Abs_SHAP",
        ascending=False

    ).reset_index(
        drop=True
    )


    pitcher_df[
        "Importance_Percent"
    ] = (

        pitcher_df[
            "Mean_Abs_SHAP"
        ]

        / pitcher_df[
            "Mean_Abs_SHAP"
        ].sum()

        * 100
    )


    print(
        "\n" + "=" * 70
    )

    print(
        "PITCHER SHAP IMPORTANCE"
    )

    print(
        "=" * 70
    )

    print(
        pitcher_df
        .to_string(
            index=False
        )
    )


    PITCHER_CSV = os.path.join(
        FINAL_DIR,
        "lstm_shap_pitcher_importance.csv"
    )


    pitcher_df.to_csv(
        PITCHER_CSV,
        index=False
    )


    plt.figure(
        figsize=(10, 7)
    )


    shap.summary_plot(
        shap_pit,
        features=feat_pit,
        feature_names=names_pit,
        max_display=min(
            12,
            len(names_pit)
        ),
        show=False
    )


    plt.title(
        "LSTM SHAP Summary – Pitcher Features"
    )

    plt.tight_layout()


    PITCHER_PNG = os.path.join(
        VIS_DIR,
        "lstm_shap_pitchers.png"
    )


    plt.savefig(
        PITCHER_PNG,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


else:

    PITCHER_CSV = None
    PITCHER_PNG = None

    print(
        "⚠️ No pitcher samples found."
    )


# ============================================================
# 13. Save raw aggregated SHAP values
# ============================================================

SHAP_NPY = os.path.join(
    FINAL_DIR,
    "lstm_shap_aggregated_values.npy"
)


np.save(
    SHAP_NPY,
    shap_aggregated
)


INDEX_CSV = os.path.join(
    FINAL_DIR,
    "lstm_shap_test_sample_indices.csv"
)


pd.DataFrame({

    "test_index":
        explain_idx,

    "feature_year":
        years[
            test_mask
        ][
            explain_idx
        ],

    "target_year":
        years[
            test_mask
        ][
            explain_idx
        ] + 1,

    "is_batter":
        is_batter,

    "is_pitcher":
        is_pitcher

}).to_csv(
    INDEX_CSV,
    index=False
)


# ============================================================
# 14. Final summary
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ FINAL LSTM SHAP COMPLETE"
)

print(
    "=" * 70
)


print(
    "\nTop 5 global SHAP features:"
)

print(
    global_importance
    .head(5)
    .to_string(
        index=False
    )
)


if BATTER_CSV is not None:

    print(
        "\nTop 5 batter SHAP features:"
    )

    print(
        batter_df
        .head(5)
        .to_string(
            index=False
        )
    )


if PITCHER_CSV is not None:

    print(
        "\nTop 5 pitcher SHAP features:"
    )

    print(
        pitcher_df
        .head(5)
        .to_string(
            index=False
        )
    )


print(
    "\n💾 Saved:"
)

print(GLOBAL_CSV)
print(GLOBAL_PNG)
print(SHAP_NPY)
print(INDEX_CSV)

if BATTER_CSV is not None:
    print(BATTER_CSV)
    print(BATTER_PNG)

if PITCHER_CSV is not None:
    print(PITCHER_CSV)
    print(PITCHER_PNG)

## 12. TFT native interpretability


In [ ]:
# ============================================================
# FINAL TFT NATIVE INTERPRETABILITY
#
# Uses the FINAL chronologically evaluated TFT model
#
# Outputs:
# 1. Static Variable Selection Network importance
# 2. Encoder Variable Selection Network importance
# 3. Temporal attention importance
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


print("=" * 70)
print("📊 FINAL TFT NATIVE INTERPRETABILITY")
print("=" * 70)


# ============================================================
# 1. Paths
# ============================================================

FINAL_DIR = "../data/processed/final"
VIS_DIR = "../data/visualizations"

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

os.makedirs(
    VIS_DIR,
    exist_ok=True
)


# ============================================================
# 2. Obtain RAW predictions from FINAL test set
# ============================================================

best_tft.eval()

prediction_output = best_tft.predict(
    test_dataloader,
    mode="raw",
    return_x=True
)


# Compatible with the return structure already used
# by the installed PyTorch Forecasting version.
raw_predictions = prediction_output[0]
x_raw = prediction_output[1]


print(
    "✅ Raw TFT predictions obtained "
    "from FINAL test set."
)


# ============================================================
# 3. Native TFT interpretation
#
# reduction="sum" gives global importance aggregated
# across the full 2024 test sample.
# ============================================================

interpretation = best_tft.interpret_output(
    raw_predictions,
    reduction="sum"
)


print(
    "✅ Native TFT interpretation extracted."
)

print(
    "Available interpretation keys:",
    list(interpretation.keys())
)


# ============================================================
# 4. Helper functions
# ============================================================

def tensor_to_1d(value):

    if torch.is_tensor(value):

        arr = (
            value
            .detach()
            .cpu()
            .numpy()
        )

    else:

        arr = np.asarray(value)

    arr = np.squeeze(arr)

    # If any additional dimensions remain,
    # aggregate all dimensions except the final one.
    while arr.ndim > 1:

        arr = arr.mean(
            axis=0
        )

    return arr.reshape(-1)


def normalize_importance(values):

    values = np.asarray(
        values,
        dtype=float
    )

    total = values.sum()

    if total > 0:

        values = (
            values
            / total
        )

    return values


# ============================================================
# 5. STATIC VARIABLE IMPORTANCE
#
# Final TFT static predictors should contain
# real handedness variables only:
# bats / throws
#
# AGE IS NOT USED.
# ============================================================

static_weights = tensor_to_1d(
    interpretation[
        "static_variables"
    ]
)


# Prefer variable names stored by the trained TFT.
static_names = list(
    getattr(
        best_tft,
        "static_variables",
        []
    )
)


# Safe fallback
if len(static_names) != len(
    static_weights
):

    static_names = [
        "bats",
        "throws"
    ]


assert len(
    static_names
) == len(
    static_weights
), (
    "Static variable name/weight "
    "length mismatch."
)


static_importance = normalize_importance(
    static_weights
)


static_df = pd.DataFrame({

    "Variable":
        static_names,

    "Importance":
        static_importance,

    "Importance_Percent":
        static_importance * 100

}).sort_values(

    "Importance",
    ascending=False

).reset_index(
    drop=True
)


print(
    "\n" + "=" * 70
)

print(
    "STATIC VARIABLE IMPORTANCE"
)

print(
    "=" * 70
)

print(
    static_df.to_string(
        index=False
    )
)


# Save static values

STATIC_CSV = os.path.join(
    FINAL_DIR,
    "tft_static_variable_importance.csv"
)

static_df.to_csv(
    STATIC_CSV,
    index=False
)


# ============================================================
# 6. Plot static variable importance
# ============================================================

plt.figure(
    figsize=(7, 4)
)

plt.barh(
    static_df["Variable"][::-1],
    static_df["Importance_Percent"][::-1]
)

plt.xlabel(
    "Global Importance (%)"
)

plt.ylabel(
    "Static Variable"
)

plt.title(
    "TFT Static Variable Selection Importance"
)

plt.tight_layout()


STATIC_PNG = os.path.join(
    VIS_DIR,
    "tft_static_vsn_importance.png"
)

plt.savefig(
    STATIC_PNG,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 7. ENCODER VARIABLE IMPORTANCE
#
# Includes:
# - final 20 baseball predictors
# - timing variables used by TFT
#
# No target variable is included as a predictor.
# ============================================================

encoder_weights = tensor_to_1d(
    interpretation[
        "encoder_variables"
    ]
)


encoder_names = list(
    getattr(
        best_tft,
        "encoder_variables",
        []
    )
)


# Fallback:
# training_dataset.reals contains the real-valued
# encoder variables in the fitted dataset.
if len(
    encoder_names
) != len(
    encoder_weights
):

    encoder_names = list(
        getattr(
            training_dataset,
            "reals",
            []
        )
    )


assert len(
    encoder_names
) == len(
    encoder_weights
), (
    f"Encoder variable name/weight mismatch: "
    f"{len(encoder_names)} names vs "
    f"{len(encoder_weights)} weights"
)


encoder_importance = normalize_importance(
    encoder_weights
)


encoder_df = pd.DataFrame({

    "Variable":
        encoder_names,

    "Importance":
        encoder_importance,

    "Importance_Percent":
        encoder_importance * 100

}).sort_values(

    "Importance",
    ascending=False

).reset_index(
    drop=True
)


print(
    "\n" + "=" * 70
)

print(
    "ENCODER VARIABLE IMPORTANCE"
)

print(
    "=" * 70
)

print(
    encoder_df.to_string(
        index=False
    )
)


ENCODER_CSV = os.path.join(
    FINAL_DIR,
    "tft_encoder_variable_importance.csv"
)

encoder_df.to_csv(
    ENCODER_CSV,
    index=False
)


# ============================================================
# 8. Plot top encoder variables
# ============================================================

TOP_N = min(
    15,
    len(encoder_df)
)

encoder_plot_df = (
    encoder_df
    .head(TOP_N)
    .sort_values(
        "Importance_Percent",
        ascending=True
    )
)


plt.figure(
    figsize=(10, 7)
)

plt.barh(
    encoder_plot_df["Variable"],
    encoder_plot_df[
        "Importance_Percent"
    ]
)

plt.xlabel(
    "Global Importance (%)"
)

plt.ylabel(
    "Encoder Variable"
)

plt.title(
    "TFT Encoder Variable Selection Importance"
)

plt.tight_layout()


ENCODER_PNG = os.path.join(
    VIS_DIR,
    "tft_encoder_vsn_importance.png"
)

plt.savefig(
    ENCODER_PNG,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. TEMPORAL ATTENTION
# ============================================================

attention_weights = tensor_to_1d(
    interpretation[
        "attention"
    ]
)


attention_importance = (
    normalize_importance(
        attention_weights
    )
)


# Final encoder contains six MLB season months:
# April–September.
if len(
    attention_importance
) == 6:

    attention_labels = [
        "April",
        "May",
        "June",
        "July",
        "August",
        "September"
    ]

else:

    attention_labels = [

        f"Encoder Step {i + 1}"

        for i in range(
            len(
                attention_importance
            )
        )
    ]


attention_df = pd.DataFrame({

    "Time_Step":
        attention_labels,

    "Importance":
        attention_importance,

    "Importance_Percent":
        attention_importance * 100

})


print(
    "\n" + "=" * 70
)

print(
    "TEMPORAL ATTENTION IMPORTANCE"
)

print(
    "=" * 70
)

print(
    attention_df.to_string(
        index=False
    )
)


ATTENTION_CSV = os.path.join(
    FINAL_DIR,
    "tft_temporal_attention_importance.csv"
)

attention_df.to_csv(
    ATTENTION_CSV,
    index=False
)


# ============================================================
# 10. Plot temporal attention
# ============================================================

plt.figure(
    figsize=(9, 5)
)

plt.bar(
    attention_df["Time_Step"],
    attention_df[
        "Importance_Percent"
    ]
)

plt.ylabel(
    "Attention Weight (%)"
)

plt.xlabel(
    "Month"
)

plt.title(
    "TFT Temporal Attention Across the MLB Season"
)

plt.xticks(
    rotation=30
)

plt.tight_layout()


ATTENTION_PNG = os.path.join(
    VIS_DIR,
    "tft_temporal_attention.png"
)

plt.savefig(
    ATTENTION_PNG,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 11. Summary
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ FINAL TFT INTERPRETABILITY COMPLETE"
)

print(
    "=" * 70
)


print(
    "\nTop static variable:"
)

print(
    static_df.iloc[0].to_dict()
)


print(
    "\nTop 5 encoder variables:"
)

print(
    encoder_df
    .head(5)
    .to_string(
        index=False
    )
)


print(
    "\nHighest-attention time step:"
)

print(
    attention_df.loc[
        attention_df[
            "Importance"
        ].idxmax()
    ].to_dict()
)


print(
    "\n💾 Saved:"
)

print(STATIC_CSV)
print(ENCODER_CSV)
print(ATTENTION_CSV)

print(STATIC_PNG)
print(ENCODER_PNG)
print(ATTENTION_PNG)

## 13. Standardised held-out test evaluation

All four models are compared using **global observation-level RMSE, MAE and R² across the same 1,106 test observations**.  
This avoids the earlier diagnostic issue in which per-batch TFT RMSE values were averaged directly.


In [ ]:
# ============================================================
# FINAL STANDARDISED TEST EVALUATION + SCATTER
#
# All metrics are calculated globally over all 1,106
# chronological test observations using the same sklearn
# definitions for fair cross-model comparison.
#
# NO model retraining.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

FINAL_DIR = "../data/processed/final"
VIS_DIR = "../data/visualizations"

os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)


# ============================================================
# 1. FINAL LSTM predictions
# ============================================================

lstm_df = pd.read_csv(
    os.path.join(
        FINAL_DIR,
        "lstm_test_predictions.csv"
    )
)

lstm_true = (
    lstm_df["actual_bwar"]
    .to_numpy(dtype=float)
)

lstm_pred = (
    lstm_df["predicted_bwar"]
    .to_numpy(dtype=float)
)


# ============================================================
# 2. FINAL TFT predictions
#
# Obtain point predictions and targets explicitly so this notebook
# works after a fresh kernel restart instead of relying on variables
# left in interactive memory.
# ============================================================

tft_prediction_output = best_tft.predict(
    test_dataloader,
    mode="prediction",
    return_y=True
)

# PyTorch Forecasting versions may return a Prediction object
# or a tuple-like structure. Handle both formats.
if hasattr(tft_prediction_output, "output"):
    tft_pred_obj = tft_prediction_output.output
    tft_y_obj = tft_prediction_output.y
else:
    tft_pred_obj = tft_prediction_output[0]
    tft_y_obj = tft_prediction_output[1]

# In some versions y is returned as (target, weight).
if isinstance(tft_y_obj, (tuple, list)):
    tft_y_obj = tft_y_obj[0]

if torch.is_tensor(tft_pred_obj):
    tft_pred = tft_pred_obj.detach().cpu().numpy().reshape(-1)
else:
    tft_pred = np.asarray(tft_pred_obj, dtype=float).reshape(-1)

if torch.is_tensor(tft_y_obj):
    tft_true = tft_y_obj.detach().cpu().numpy().reshape(-1)
else:
    tft_true = np.asarray(tft_y_obj, dtype=float).reshape(-1)

assert len(lstm_true) == 1106
assert len(tft_true) == 1106

assert np.allclose(
    lstm_true,
    tft_true,
    atol=1e-6
)

print(
    "✅ LSTM and TFT evaluated on "
    "the same 1,106 test targets."
)


# ============================================================
# 3. Common GLOBAL metric function
# ============================================================

def global_metrics(y_true, y_pred):

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2)
    }


lstm_metrics = global_metrics(
    lstm_true,
    lstm_pred
)

tft_metrics_global = global_metrics(
    tft_true,
    tft_pred
)


print("\n" + "=" * 70)
print("STANDARDISED GLOBAL TEST METRICS")
print("=" * 70)

print(
    "LSTM | "
    f"RMSE={lstm_metrics['rmse']:.6f} | "
    f"MAE={lstm_metrics['mae']:.6f} | "
    f"R²={lstm_metrics['r2']:.6f}"
)

print(
    "TFT  | "
    f"RMSE={tft_metrics_global['rmse']:.6f} | "
    f"MAE={tft_metrics_global['mae']:.6f} | "
    f"R²={tft_metrics_global['r2']:.6f}"
)


# ============================================================
# 4. Save FINAL TFT point predictions
# ============================================================

tft_pred_df = pd.DataFrame({
    "target_year": 2024,
    "actual_bwar": tft_true,
    "predicted_bwar": tft_pred
})

TFT_PRED_PATH = os.path.join(
    FINAL_DIR,
    "tft_test_predictions.csv"
)

tft_pred_df.to_csv(
    TFT_PRED_PATH,
    index=False
)


# ============================================================
# 5. Save standardised TFT metrics
#
# Keep original trainer.test metrics for audit.
# Do NOT overwrite them.
# ============================================================

GLOBAL_TFT_METRIC_PATH = os.path.join(
    FINAL_DIR,
    "tft_test_metrics_global.json"
)

with open(
    GLOBAL_TFT_METRIC_PATH,
    "w"
) as f:

    json.dump(
        tft_metrics_global,
        f,
        indent=2
    )


# ============================================================
# 6. FINAL four-model comparison
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [
        "Ridge Regression",
        "LSTM",
        "TFT",
        "Random Forest"
    ],

    "RMSE": [
        ridge_test_metrics["rmse"],
        lstm_metrics["rmse"],
        tft_metrics_global["rmse"],
        rf_test_metrics["rmse"]
    ],

    "MAE": [
        ridge_test_metrics["mae"],
        lstm_metrics["mae"],
        tft_metrics_global["mae"],
        rf_test_metrics["mae"]
    ],

    "R2": [
        ridge_test_metrics["r2"],
        lstm_metrics["r2"],
        tft_metrics_global["r2"],
        rf_test_metrics["r2"]
    ]
})

comparison_df = (
    comparison_df
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("FINAL STANDARDISED MODEL COMPARISON")
print("=" * 70)

print(
    comparison_df.to_string(
        index=False
    )
)

comparison_df.to_csv(
    os.path.join(
        FINAL_DIR,
        "final_model_comparison.csv"
    ),
    index=False
)


# ============================================================
# 7. Scatter plot
# ============================================================

all_values = np.concatenate([
    lstm_true,
    lstm_pred,
    tft_true,
    tft_pred
])

plot_min = np.floor(
    all_values.min()
)

plot_max = np.ceil(
    all_values.max()
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.5),
    sharex=True,
    sharey=True
)


plot_models = [

    (
        "LSTM",
        lstm_true,
        lstm_pred,
        lstm_metrics
    ),

    (
        "Temporal Fusion Transformer (TFT)",
        tft_true,
        tft_pred,
        tft_metrics_global
    )
]


for ax, (
    model_name,
    actual,
    predicted,
    metrics
) in zip(
    axes,
    plot_models
):

    ax.scatter(
        actual,
        predicted,
        alpha=0.35,
        s=18,
        edgecolors="none"
    )

    ax.plot(
        [plot_min, plot_max],
        [plot_min, plot_max],
        linestyle="--",
        linewidth=1.5,
        label="Perfect prediction"
    )

    ax.set_xlim(
        plot_min,
        plot_max
    )

    ax.set_ylim(
        plot_min,
        plot_max
    )

    ax.set_title(
        model_name,
        fontsize=12
    )

    ax.set_xlabel(
        "Observed 2024 bWAR",
        fontsize=11
    )

    ax.tick_params(
        axis="both",
        labelsize=10
    )

    ax.grid(
        alpha=0.25
    )

    metric_text = (
        f"RMSE = {metrics['rmse']:.3f}\n"
        f"MAE = {metrics['mae']:.3f}\n"
        f"$R^2$ = {metrics['r2']:.3f}"
    )

    ax.text(
        0.05,
        0.95,
        metric_text,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="top",
        bbox=dict(
            boxstyle="round",
            alpha=0.15
        )
    )

    ax.legend(
        loc="lower right",
        fontsize=9
    )


axes[0].set_ylabel(
    "Predicted 2024 bWAR",
    fontsize=11
)

plt.tight_layout()


FINAL_SCATTER_PATH = os.path.join(
    VIS_DIR,
    "model_comparison_scatter_final.png"
)

plt.savefig(
    FINAL_SCATTER_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\n✅ Saved:")
print(TFT_PRED_PATH)
print(GLOBAL_TFT_METRIC_PATH)
print(
    os.path.join(
        FINAL_DIR,
        "final_model_comparison.csv"
    )
)
print(FINAL_SCATTER_PATH)


## 14. Target descriptive statistics


In [ ]:
import pandas as pd
import numpy as np

def summarise_target(name, values):
    values = np.asarray(values).reshape(-1)
    return {
        "Sample": name,
        "N": len(values),
        "Mean": np.mean(values),
        "SD": np.std(values, ddof=1),
        "Median": np.median(values),
        "Min": np.min(values),
        "Max": np.max(values)
    }

summary_stats = pd.DataFrame([
    summarise_target("Full dataset", y),
    summarise_target("Training", y_train),
    summarise_target("Validation", y_val),
    summarise_target("Test", y_test)
])

print(summary_stats.round(3))

## Reference results from the submitted dissertation

When run with the dissertation data and environment, the final held-out 2024 results should reproduce approximately:

| Model | RMSE | MAE | R² |
|---|---:|---:|---:|
| Ridge Regression | 1.373442 | 0.992555 | 0.227437 |
| LSTM | 1.380756 | 0.988436 | 0.219187 |
| TFT | 1.385182 | 0.996271 | 0.214173 |
| Random Forest | 1.402113 | 1.004001 | 0.194846 |

These values are provided as a reproducibility check; the comparison cell above calculates metrics from model predictions rather than hard-coding them.
